In [11]:
import os
import pandas as pd

In [16]:
df = pd.read_excel(
    "../data/processed/data_cleaned_for_import.xlsx",
    sheet_name="contracts"
)

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 540 entries, 0 to 539
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   contract_id                     540 non-null    int64         
 1   counterpart_id                  540 non-null    int64         
 2   date                            540 non-null    datetime64[us]
 3   number                          540 non-null    int64         
 4   quantity                        315 non-null    float64       
 5   delivery_due                    417 non-null    datetime64[us]
 6   total_by_ctr                    337 non-null    float64       
 7   total_fact_payments_lookup      540 non-null    float64       
 8   price_without_vat               540 non-null    float64       
 9   vat                             540 non-null    float64       
 10  price_with_vat                  537 non-null    float64       
 11  shipped_by_suppli

In [18]:
df.head(2)

,contract_id,counterpart_id,date,number,quantity,delivery_due,total_by_ctr,total_fact_payments_lookup,price_without_vat,vat,price_with_vat,shipped_by_supplier_mt,received_on_wh_mt,contract_group_id,contract_type_id,vessel_loose_ctrgroups_lookup,vessel_id_ctrgroups_lookup,received_total_payments_lookup,paid_total_payments_lookup
0,1,3,2023-07-31,7429670,28600.0,2023-11-18,675052950.0,6.741610e+08,23603.25,0.0,23603.25,28562.218,28562.218,94,36,1,NaN,6.741610e+08,0.0
1,2,2,2024-01-21,7496406,1900.0,2024-02-19,81395050.0,8.151414e+07,42839.50,0.0,42839.50,1902.780,1902.780,97,36,9,9.0,8.151414e+07,0.0


In [12]:
# Check all sheets
file = "../data/processed/data_cleaned_for_import.xlsx"

excel = pd.ExcelFile(file)

print(excel.sheet_names)

['payments', 'vessel_allocation', 'contracts', 'contract_groups', 'contract_types', 'counterparts', 'vessels', 'projects']


In [19]:
input_file = "../data/processed/data_cleaned_for_import.xlsx"
output_folder = "../data/csv"

os.makedirs(output_folder, exist_ok=True)

excel = pd.ExcelFile(input_file)

for sheet in excel.sheet_names:

    df = pd.read_excel(
        excel,
        sheet_name=sheet
    )

    # Названия колонок
    df.columns = (
        df.columns
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", "_", regex=True)
    )

    # Еще раз убрать пробелы в текстовых колонках
    text_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns
        
    for col in text_columns:
        df[col] = df[col].str.strip()

    # Проверка
    print("\n", sheet)
    print(df.dtypes)
    print(df.isna().sum())

    # Сохранить в csv
    df.to_csv(
    f"{output_folder}/{sheet}.csv",
    index=False,
    encoding="utf-8",
    sep=",",
    decimal="."
    )
    
    print(f"{sheet} saved")

C:\Users\alinaaleks\AppData\Local\Temp\ipykernel_8416\1997723481.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(



 payments
payment_id                           int64
internal_number                      int64
payment_date                datetime64[us]
contract_id                          int64
counterpart_id                       int64
received_rub                       float64
paid_rub                           float64
received_minus_paid_calc           float64
operation_type                         str
dtype: object
payment_id                  0
internal_number             0
payment_date                1
contract_id                 0
counterpart_id              0
received_rub                1
paid_rub                    1
received_minus_paid_calc    1
operation_type              0
dtype: int64
payments saved

 vessel_allocation
vessel_allocation_id                     int64
vessel_id                                int64
contract_id                              int64
quantity_allocated                     float64
quantity_planned                       float64
contract_group_ctr_lookup          

C:\Users\alinaaleks\AppData\Local\Temp\ipykernel_8416\1997723481.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(
C:\Users\alinaaleks\AppData\Local\Temp\ipykernel_8416\1997723481.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-

# на будущее
# Даты привести к ISO
date_columns = [
        col for col in df.columns
            if any(x in col for x in ["date", "due"])
    ]
    
    for col in date_columns:
        df[col] = (
            pd.to_datetime(
                df[col],
                errors="coerse"
            )
                .dt.strftime("%Y-%m-%d")
        )